In [2]:
import requests
import pandas as pd

In [3]:
URL = "https://stephen-king-api.onrender.com/api/books" 
response = requests.get(URL)

data = response.json()
books = data["data"] 

len(books)

64

In [4]:
df = pd.DataFrame(books)

# df.head()
# Check the structure:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          64 non-null     int64 
 1   Year        64 non-null     int64 
 2   Title       64 non-null     object
 3   handle      64 non-null     object
 4   Publisher   64 non-null     object
 5   ISBN        64 non-null     object
 6   Pages       64 non-null     int64 
 7   Notes       64 non-null     object
 8   created_at  64 non-null     object
 9   villains    64 non-null     object
dtypes: int64(3), object(7)
memory usage: 5.1+ KB


In [5]:
book_df = df[ [ "id", "Year", "Title", "Publisher", "ISBN", "Pages", "villains" ] ].copy() 

book_df.head()

,id,Year,Title,Publisher,ISBN,Pages,villains
0,1,1974,Carrie,Doubleday,978-0-385-08695-0,199,"[{'name': 'Tina Blake', 'url': 'https://stephe..."
1,2,1975,Salem's Lot,Doubleday,978-0-385-00751-1,439,"[{'name': 'Kurt Barlow', 'url': 'https://steph..."
2,3,1977,The Shining,Doubleday,978-0-385-12167-5,447,"[{'name': 'Horace M. Derwent', 'url': 'https:/..."
3,4,1977,Rage,Signet Books,978-0-451-07645-8,211,[]
4,5,1978,The Stand,Doubleday,978-0-385-12168-2,823,"[{'name': 'Donald Merwin Elbert', 'url': 'http..."


In [6]:
#Calculate number of villains per book

In [7]:
book_df["villain_names"] = book_df["villains"].apply( lambda x: [v["name"] for v in x] )

book_df[["Title", "villain_names"]].head()

,Title,villain_names
0,Carrie,"[Tina Blake, Cindi, Myra Crewes, Billy deLois,..."
1,Salem's Lot,"[Kurt Barlow, Richard Straker]"
2,The Shining,"[Horace M. Derwent, Delbert Grady, Jack Torrance]"
3,Rage,[]
4,The Stand,"[Donald Merwin Elbert, Randall Flagg, Lloyd He..."


In [8]:
book_df["villain_count"] = book_df["villain_names"].apply(len)

book_df[ ["Title", "Year", "Pages", "villain_count"] ].sort_values( "villain_count", ascending=False ).head(10)

,Title,Year,Pages,villain_count
0,Carrie,1974,199,17
18,It,1986,1138,15
33,The Dark Tower IV: Wizard and Glass,1997,787,6
4,The Stand,1978,823,6
6,The Dead Zone,1979,428,6
11,The Dark Tower: The Gunslinger,1982,224,4
13,Pet Sematary,1983,374,4
41,The Dark Tower VII: The Dark Tower,2004,845,4
7,Firestarter,1980,426,3
26,Gerald's Game,1992,352,3


In [9]:
book_df["villain_density"] = (
    book_df["villain_count"] / book_df["Pages"]
) * 100


book_df[
    [
        "Title",
        "Pages",
        "villain_count",
        "villain_density"
    ]
].sort_values(
    "villain_density",
    ascending=False
).head(15)

,Title,Pages,villain_count,villain_density
0,Carrie,199,17,8.542714
11,The Dark Tower: The Gunslinger,224,4,1.785714
6,The Dead Zone,428,6,1.401869
18,It,1138,15,1.318102
13,Pet Sematary,374,4,1.069519
10,The Running Man,219,2,0.913242
26,Gerald's Game,352,3,0.852273
14,Cycle of the Werewolf,127,1,0.787402
33,The Dark Tower IV: Wizard and Glass,787,6,0.762389
4,The Stand,823,6,0.729040


In [10]:
book_df["danger_score"] = ( 
    book_df["villain_count"] * 10 + book_df["villain_density"] * 5 + book_df["Pages"] / 100 
)

In [29]:
dangerous_books = book_df.sort_values( "danger_score", ascending=False )

dangerous_books[ [ 
    "Title", 
    "Year",
    "Pages",
    "villain_count",
    "villain_density",
    "danger_score" ]
    ].head(15)

,Title,Year,Pages,villain_count,villain_density,danger_score
0,Carrie,1974,199,17,8.542714,214.703568
18,It,1986,1138,15,1.318102,167.970510
4,The Stand,1978,823,6,0.729040,71.875200
33,The Dark Tower IV: Wizard and Glass,1997,787,6,0.762389,71.681944
6,The Dead Zone,1979,428,6,1.401869,71.289346
11,The Dark Tower: The Gunslinger,1982,224,4,1.785714,51.168571
41,The Dark Tower VII: The Dark Tower,2004,845,4,0.473373,50.816864
13,Pet Sematary,1983,374,4,1.069519,49.087594
47,Under the Dome,2009,1074,3,0.279330,42.136648
2,The Shining,1977,447,3,0.671141,37.825705


In [30]:
villains_df = book_df[
    ["id", "Title", "Year", "villains"]
].explode("villains")

# villains_df.head(40)

In [31]:
villains_df["villain_name"] = villains_df[
    "villains"
].apply(
    lambda x: x["name"] if isinstance(x, dict) else None
)

villains_df["villain_url"] = villains_df[
    "villains"
].apply(
    lambda x: x["url"] if isinstance(x, dict) else None
)

In [33]:
villains_df = villains_df[
    [
        "id",
        "Title",
        "Year",
        "villain_name",
        "villain_url"
    ]
]

# villains_df

villains_df.iloc[0]["villain_url"]

'https://stephen-king-api.onrender.com/api/villain/4'